In [2]:
import pandas as pd
import fitz
import re
from tqdm import tqdm

In [5]:
def clean_text(text):
    """Очистка текста от лишних переносов и мусора"""
    # Склеиваем слова, разделенные переносом на новую строку (например, корпо-ративный -> корпоративный)
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    # Заменяем множественные пробелы и переносы на одиночные
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [6]:
def parse_and_split_pdf(pdf_path, output_txt_path, chunk_size=800, overlap=150):
    """Парсинг PDF и нарезка на чанки с сохранением номеров страниц"""
    doc = fitz.open(pdf_path)
    all_chunks = []
    
    print(f"Начало обработки PDF. Всего страниц: {len(doc)}")
    
    for page_num in tqdm(range(len(doc))):
        page = doc[page_num]
        # Извлекаем текст именно с этой страницы
        page_text = page.get_text("text")
        cleaned_page_text = clean_text(page_text)
        
        if not cleaned_page_text:
            continue
            
        # Нарезаем текст страницы на пересекающиеся куски (chunks)
        # Перекрытие (overlap) нужно, чтобы важная мысль не оборвалась на полуслове
        start = 0
        while start < len(cleaned_page_text):
            end = start + chunk_size
            chunk = cleaned_page_text[start:end]
            
            # Формируем кусок с метаданными. Модель за счет этого будет знать контекст страницы
            meta_chunk = f"[СТРАНИЦА {page_num + 1}] {chunk}"
            all_chunks.append(meta_chunk)
            
            # Сдвигаем окно вперед с учетом перекрытия
            start += (chunk_size - overlap)

    # Сохраняем результат в файл, разделяя куски нашим спецсимволом
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write("---CHUNK_SPLIT---".join(all_chunks))
        
    print(f"Успешно создано {len(all_chunks)} кусков текста. Сохранено в {output_txt_path}")

In [7]:
parse_and_split_pdf('../data/Руководство для ментора.pdf', '../data/mentor_basics_1.txt')

Начало обработки PDF. Всего страниц: 23


100%|██████████| 23/23 [00:00<00:00, 236.36it/s]

Успешно создано 83 кусков текста. Сохранено в ../data/mentor_basics_1.txt


In [ ]:
print("Создание новой базы знаний...")
with open('ethic_code.txt', "r", encoding="utf-8") as f:
    # Просто читаем уже готовые чанки со страницами из файла
    chunks = f.read().split("---CHUNK_SPLIT---")

Создание новой базы знаний...
